# System Design Exercise: Inference for 1M Concurrent Users

**Section 5 | 15 minutes**

Apply everything you learned in the last 90 minutes. This is a thinking exercise:
no GPUs required, just the equations and intuitions you've built up.

## The Challenge

You are the **infra lead** at a Series C startup. The CEO just closed a deal:
a global chatbot serving **1M concurrent users**.

**Constraints:**
- Time to First Token (TTFT) < 500ms
- Inter-Token Latency (ITL) < 80ms
- Budget: minimize cost (you report to the CFO weekly)
- Model: 70B parameter LLM (your fine-tuned variant)
- Average context length: 4096 tokens
- Average generation length: 512 tokens

Design the inference infrastructure from scratch.

## The 10-Part Framework

Use these 10 dimensions to structure your design:

| # | Dimension | Key Question |
|---|-----------|-------------|
| 1 | **Requirements** | What are the SLOs? What's the traffic pattern? |
| 2 | **Model** | What architecture? What precision? Quantized? |
| 3 | **Memory** | How much VRAM per replica? KV cache budget? |
| 4 | **Hardware** | Which GPU? How many per node? |
| 5 | **Parallelism** | TP, PP, or both? Disaggregated prefill/decode? |
| 6 | **Serving** | Which engine? Continuous batching config? |
| 7 | **Caching** | Prefix caching? Semantic cache? KV offloading? |
| 8 | **Monitoring** | What metrics? Alerts? SLO tracking? |
| 9 | **Scaling** | Autoscaling policy? Multi-region? |
| 10 | **Failures** | What breaks? Graceful degradation? |

In [ ]:
# YOUR DESIGN: Fill in each dimension
# Spend 5 minutes thinking, then compare with your neighbor

my_design = {
    '1_requirements': {
        'slo_ttft_ms': None,
        'slo_itl_ms': None,
        'concurrent_users': None,
        'requests_per_second': None,  # estimate from concurrency + avg session
    },
    '2_model': {
        'params_billions': None,
        'precision': None,  # fp16, int8, int4?
        'architecture': None,
    },
    '3_memory': {
        'model_weights_gb': None,  # params * bytes_per_param
        'kv_cache_per_request_mb': None,
        'max_batch_kv_gb': None,
    },
    '4_hardware': {
        'gpu_type': None,  # A100-80GB? H100? A10G?
        'vram_per_gpu_gb': None,
        'gpus_per_node': None,
    },
    '5_parallelism': {
        'tensor_parallel_degree': None,
        'pipeline_parallel_degree': None,
        'disaggregated_prefill_decode': None,  # True/False
    },
    '6_serving': {
        'engine': None,  # vLLM, SGLang, TRT-LLM?
        'max_batch_size': None,
        'scheduling': None,  # continuous batching config
    },
    '7_caching': {
        'prefix_caching': None,
        'semantic_cache': None,
        'kv_offload_to_cpu': None,
    },
    '8_monitoring': {
        'key_metrics': [],
        'alerting_thresholds': {},
    },
    '9_scaling': {
        'autoscaling_metric': None,
        'min_replicas': None,
        'max_replicas': None,
        'regions': [],
    },
    '10_failures': {
        'single_gpu_failure': None,
        'full_node_failure': None,
        'traffic_spike_2x': None,
    },
}

# Print your design
import json
print(json.dumps(my_design, indent=2, default=str))

## Hints

Use these derivations from earlier sections:

**Memory equation (Section 1):**
```
Model weights = params * bytes_per_param
70B at FP16 = 70e9 * 2 bytes = 140 GB
70B at INT4  = 70e9 * 0.5 bytes = 35 GB
```

**KV cache per request (Section 2):**
```
KV per token = 2 * num_layers * num_heads * head_dim * bytes_per_param
Llama 70B: 2 * 80 * 8 * 128 * 2 = 2.56 MB per 1K tokens (GQA, FP16)
At 4096 context: ~10.5 MB per request
```

**Fleet sizing:**
```
Tokens/sec per GPU (decode) ~ 2000-4000 (depends on batch size)
Each user generates ~512 tokens at ~40 tok/s perceived
Active generation time per request ~ 12.8 seconds
```

**Cost reference (on-demand):**
```
A100-80GB: ~$3.50/hr (cloud)
H100-80GB: ~$5.50/hr (cloud)
A10G-24GB:  ~$1.50/hr (cloud)
```

In [ ]:
def compute_fleet_size(model_params_b, precision_bytes, concurrent_users,
                       avg_context_len, gpu_memory_gb, tp_degree=1):
    """
    Estimate minimum GPU fleet size for a target concurrency.
    
    Returns dict with model memory, KV budget, batch capacity, and fleet size.
    """
    # Model weights across TP shards
    model_memory_gb = (model_params_b * 1e9 * precision_bytes) / 1e9
    memory_per_gpu = model_memory_gb / tp_degree
    
    # Available for KV cache (reserve 20% for activations + overhead)
    kv_budget_gb = (gpu_memory_gb * 0.8) - memory_per_gpu
    
    # KV cache per request (approximation for GQA models)
    # 2 * layers * kv_heads * head_dim * bytes * seq_len
    # Simplified: ~2.5 MB per 1K tokens for 70B GQA at FP16
    kv_per_request_gb = (2.5 * avg_context_len / 1000) / 1024
    
    # Max concurrent requests per TP group
    batch_capacity = int(kv_budget_gb / kv_per_request_gb) if kv_per_request_gb > 0 else 0
    
    # Fleet size (number of TP groups needed)
    if batch_capacity > 0:
        tp_groups_needed = -(-concurrent_users // batch_capacity)  # ceiling division
    else:
        tp_groups_needed = float('inf')
    
    total_gpus = tp_groups_needed * tp_degree
    
    return {
        'model_memory_gb': round(model_memory_gb, 1),
        'memory_per_gpu_gb': round(memory_per_gpu, 1),
        'kv_budget_per_gpu_gb': round(kv_budget_gb, 1),
        'kv_per_request_gb': round(kv_per_request_gb, 4),
        'batch_capacity_per_tp_group': batch_capacity,
        'tp_groups_needed': tp_groups_needed,
        'total_gpus': total_gpus,
    }

# Example: 70B at INT4 on H100-80GB with TP=2
result = compute_fleet_size(
    model_params_b=70, precision_bytes=0.5,
    concurrent_users=1_000_000, avg_context_len=4096,
    gpu_memory_gb=80, tp_degree=2
)
for k, v in result.items():
    print(f'{k}: {v}')

In [ ]:
def compute_cost_per_conversation(fleet_size, instance_cost_per_hr,
                                   gpus_per_instance, conversations_per_hour):
    """
    Estimate cost per conversation given fleet and traffic.
    
    Args:
        fleet_size: total GPUs
        instance_cost_per_hr: cost per instance (not per GPU)
        gpus_per_instance: GPUs per instance
        conversations_per_hour: total conversations across fleet
    """
    num_instances = fleet_size / gpus_per_instance
    total_cost_per_hour = num_instances * instance_cost_per_hr
    cost_per_conversation = total_cost_per_hour / conversations_per_hour
    
    return {
        'num_instances': int(num_instances),
        'total_cost_per_hour_usd': round(total_cost_per_hour, 2),
        'cost_per_conversation_usd': round(cost_per_conversation, 6),
        'cost_per_1k_conversations_usd': round(cost_per_conversation * 1000, 2),
        'monthly_cost_usd': round(total_cost_per_hour * 24 * 30, 0),
    }

# Example: Using fleet from above
cost = compute_cost_per_conversation(
    fleet_size=result['total_gpus'],
    instance_cost_per_hr=11.0,  # p5.2xlarge (8xH100) ~ $11/hr equivalent per 2-GPU
    gpus_per_instance=8,
    conversations_per_hour=2_000_000  # 1M concurrent, avg 2 convos/hr each
)
for k, v in cost.items():
    print(f'{k}: {v}')

## Discussion Questions

Pair up with your neighbor. Spend 3 minutes on these:

1. **Quantization tradeoff:** If you go from FP16 to INT4, you cut memory 4x but may lose 2-3% accuracy. When is this acceptable? When is it not?

2. **Disaggregated prefill/decode:** You could separate prefill (compute-bound) from decode (memory-bound) onto different hardware. What's the operational cost of this complexity vs. the efficiency gain?

3. **Caching economics:** If 30% of your queries share a common system prompt prefix, how much memory does prefix caching save? Is a semantic cache (embedding similarity) worth the latency overhead?

4. **Failure blast radius:** One H100 node dies (8 GPUs). With TP=4, you lose 2 serving replicas. How do you design so users never notice? What's the cost of that redundancy?

5. **The 3 AM question:** Traffic drops to 50K concurrent at 3 AM. Your autoscaler scales down. At 9 AM, traffic spikes to 1.5M. Cold-start for a 70B model takes 45 seconds. How do you handle the ramp?

In [ ]:
# ============================================================
# SAMPLE SOLUTION (review after your own attempt)
# ============================================================

sample_solution = {
    '1_requirements': {
        'slo_ttft_ms': 500,
        'slo_itl_ms': 80,
        'concurrent_users': 1_000_000,
        'requests_per_second': 83_333,  # 1M users, avg 12s generation, ~83K active
        'regions': ['us-east-1', 'eu-west-1', 'ap-northeast-1'],
    },
    '2_model': {
        'params_billions': 70,
        'precision': 'INT4 (AWQ)',  # 4x memory savings, <1% quality loss on chatbot
        'architecture': 'Llama-3 70B variant with GQA (8 KV heads)',
    },
    '3_memory': {
        'model_weights_gb': 35,  # 70B * 0.5 bytes
        'kv_cache_per_request_mb': 10.5,  # 4096 tokens * 2.56 MB/1K
        'max_batch_kv_gb': 5.4,  # 512 concurrent requests * 10.5MB
        'total_per_gpu_gb': 'weights(17.5/tp2) + KV(5.4) + overhead(8) = ~31 GB',
    },
    '4_hardware': {
        'gpu_type': 'H100-80GB SXM',
        'vram_per_gpu_gb': 80,
        'gpus_per_node': 8,
        'reason': '80GB allows large batches; NVLink for TP communication',
    },
    '5_parallelism': {
        'tensor_parallel_degree': 2,  # 70B INT4 fits in 2xH100
        'pipeline_parallel_degree': 1,  # avoid PP latency penalty
        'disaggregated_prefill_decode': True,  # separate P and D pools
        'reason': 'Prefill is compute-bound (use fewer GPUs harder), decode is memory-bound (maximize batch)',
    },
    '6_serving': {
        'engine': 'vLLM with PagedAttention',
        'max_batch_size': 512,  # per TP group
        'scheduling': 'Continuous batching, chunked prefill (512 token chunks)',
        'reason': 'PagedAttention eliminates KV fragmentation, chunked prefill controls TTFT',
    },
    '7_caching': {
        'prefix_caching': True,  # system prompt shared across all requests
        'semantic_cache': False,  # latency overhead not worth it for chatbot
        'kv_offload_to_cpu': 'Only for overflow/degraded mode',
        'prefix_savings': '~2GB shared KV for 2048-token system prompt',
    },
    '8_monitoring': {
        'key_metrics': [
            'P50/P95/P99 TTFT and ITL',
            'GPU memory utilization (%)',
            'Batch size distribution',
            'Queue depth (requests waiting)',
            'Token throughput (tok/s/GPU)',
            'Request success rate',
        ],
        'alerting': 'Page if P99 TTFT > 800ms for 2 min, or success rate < 99.5%',
    },
    '9_scaling': {
        'autoscaling_metric': 'Queue depth + GPU memory utilization',
        'min_replicas': 500,  # floor for 50K off-peak users
        'max_replicas': 2500,  # ceiling for 1.5M spike
        'scale_up_policy': 'Pre-warm 20% buffer; scale on queue > 100 for 30s',
        'multi_region': '3 regions, geo-routed, each independently scaled',
    },
    '10_failures': {
        'single_gpu_failure': 'TP group becomes unhealthy, drain + replace (30s)',
        'full_node_failure': 'Lose 4 TP groups (TP=2, 8 GPUs). Traffic shifts to remaining nodes.',
        'traffic_spike_2x': 'Buffer pool (20% idle warm) absorbs first 2 min, then autoscaler catches up',
        'graceful_degradation': 'At capacity: reduce max_tokens, increase chunked prefill, queue overflow to secondary region',
    },
}

# Fleet estimate with solution params
solution_fleet = compute_fleet_size(
    model_params_b=70, precision_bytes=0.5,
    concurrent_users=1_000_000, avg_context_len=4096,
    gpu_memory_gb=80, tp_degree=2
)

print('=== FLEET ESTIMATE ===')
for k, v in solution_fleet.items():
    print(f'  {k}: {v}')

print(f'\n=== COST ESTIMATE ===')
solution_cost = compute_cost_per_conversation(
    fleet_size=solution_fleet['total_gpus'],
    instance_cost_per_hr=32.77,  # p5.48xlarge (8xH100)
    gpus_per_instance=8,
    conversations_per_hour=3_600_000  # 1M concurrent * ~3.6 convos/hr
)
for k, v in solution_cost.items():
    print(f'  {k}: {v}')

print(f'\n=== FULL SOLUTION ===')
print(json.dumps(sample_solution, indent=2))

## Wrap-Up: Resources

**You've completed the workshop.** In 2 hours, you went from memory equations to designing a system serving 1M users.

### Continue Learning

**Full repository (50 modules, all open-source):**
- [github.com/harshuljain13/llm-inference-at-scale](https://github.com/harshuljain13/llm-inference-at-scale)
- Every module has a Colab notebook you can run for free on T4

**Chapter highlights:**
- Ch00-03: Foundations (transformer memory, GPU architecture, KV cache)
- Ch04-06: Optimizations (batching, engines, quantization + MoE)
- Ch07-10: Production (serving infra, case studies, system designs)

### What's Next

- **Manning 'In Action' book** (coming 2027): structured version of this content with exercises and production patterns
- **LLM Inferencing Maven Cohort** with Abi Aryan (Jul 18 - Sep 12): hands-on group learning with industry practitioners

### Connect

- GitHub: [harshuljain13](https://github.com/harshuljain13)
- LinkedIn: [linkedin.com/in/hjain1393](https://www.linkedin.com/in/hjain1393/)

Thank you for spending your afternoon with us. Now go build something that scales. 🚀